# 第 10 章代码实验：并发、并行与异步

以当前项目 CPython 3.12.13 为基线。先阅读讲义对应部分，再逐题完成：读代码、不运行 → 填 prediction → 运行 → 填 observation → 即时检查 → 解释。两个字典都要把整个 `"__未填写__"`（包括引号）替换为实际 Python 值；字段编号对应具体表达式或时刻，不能删除或重复。

预测错误时保留最初预测，只纠正实际观察并解释原因。检查器只要求预测已填写，核对的是实际观察，不把预测错误当作必须抹掉的记录。不要预览检查文件中的判定数据代替推理。

实验 1～24 为观察题，25～28 为实现题。每题只有自己的即时检查，不依赖其他题先完成。第 27、28 题的异步实现使用 `await check_async_experiment(...)`；其他题使用普通 `check_experiment(...)`。整章末尾另有异步复查入口。

线程和进程实验使用事件、屏障及明确的数据约束，不把随机调度顺序、PID 或精确耗时作为答案。实验 13、14 把可见源码保存到临时脚本，用当前解释器启动并清理，解决 Windows/Jupyter 下 spawn 的可导入入口要求。循环中的同步示例不要擅自改成无上限阻塞等待。

Jupyter 已有事件循环，异步观察单元格使用顶层 await，不使用嵌套 asyncio.run。Notebook 本身不需要联网，不访问真实账号、数据库或浏览器。自写代码若形成无响应死锁，需要中断或重启内核并检查等待关系，不能靠反复重跑叠加后台任务。

## 实验 1：线程对象、启动与结束

按执行阶段预测 events 和接口调用结果。创建线程对象之后，先记录状态，再启动；不根据输出先后猜测调度。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_01 = {
    "01. start 前的 before": "__未填写__",
    "02. join 后的 events": "__未填写__",
    "03. start() 的返回结果": "__未填写__",
    "04. join 后 thread.is_alive()": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from threading import Thread

events = []
def work():
    events.append("work")

thread = Thread(target=work)
before = events.copy()
returned = thread.start()
thread.join()
actual_01 = (before, events.copy(), returned, thread.is_alive())
print(*actual_01, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_01 = {
    "01. start 前的 before": "__未填写__",
    "02. join 后的 events": "__未填写__",
    "03. start() 的返回结果": "__未填写__",
    "04. join 后 thread.is_alive()": "__未填写__",
}

### 实验 1 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 1)

### 实验 1 原因解释

分别解释构造对象、start 和 join 的责任。为什么 target 接收函数对象，而不是提前调用函数？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 2：线程中的共享列表与局部重新绑定

画出 records 与工作函数 items 的绑定。线程先修改传入对象，再把自己的局部名字绑定到另一列表；结果通过队列返回。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_02 = {
    "01. join 后 records": "__未填写__",
    "02. 工作函数放入 results 的对象内容": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from queue import Queue
from threading import Thread

records = []
results = Queue()
def work(items):
    items.append("written")
    items = ["local only"]
    results.put(items)

thread = Thread(target=work, args=(records,))
thread.start()
thread.join()
actual_02 = (records.copy(), results.get_nowait())
print(*actual_02, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_02 = {
    "01. join 后 records": "__未填写__",
    "02. 工作函数放入 results 的对象内容": "__未填写__",
}

### 实验 2 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 2)

### 实验 2 原因解释

指出哪一步修改共享对象，哪一步只重新绑定局部形参。如果多个线程都访问 records，当前代码还需要哪些约束？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 3：用屏障控制丢失更新

Barrier 要求两个工作线程都完成读取后才进入写回。跟踪每个线程的局部 old 与共享字典，不预测线程身份或打印顺序。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_03 = {
    "01. 两线程结束后的 state['count']": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from threading import Barrier, Thread

state = {"count": 0}
both_read = Barrier(2, timeout=5)
def increase():
    old = state["count"]
    both_read.wait()
    state["count"] = old + 1

threads = [Thread(target=increase) for _ in range(2)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
actual_03 = (state["count"],)
print(*actual_03, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_03 = {
    "01. 两线程结束后的 state['count']": "__未填写__",
}

### 实验 3 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 3)

### 实验 3 原因解释

列出读、等、写三步交错。为什么当前 CPython 有 GIL，也不保护这项完整更新？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 4：锁保护完整更新

四个工作线程各进行 50 次更新，所有更新都使用同一把锁。记录最终状态及锁的状态。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_04 = {
    "01. join 后的 state['count']": "__未填写__",
    "02. join 后 lock.locked()": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from threading import Lock, Thread

state = {"count": 0}
lock = Lock()
def increase():
    for _ in range(50):
        with lock:
            state["count"] = state["count"] + 1

threads = [Thread(target=increase) for _ in range(4)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
actual_04 = (state["count"], lock.locked())
print(*actual_04, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_04 = {
    "01. join 后的 state['count']": "__未填写__",
    "02. join 后 lock.locked()": "__未填写__",
}

### 实验 4 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 4)

### 实验 4 原因解释

如果只把写回那行放进锁，却在锁外读取旧值，为什么约束会变化？with 如何处理异常退出？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 5：普通锁与可重入锁

同一线程持锁后再次尝试取得锁。使用 blocking=False，实验不会故意进入死锁。成功重复取得时有配对释放。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_05 = {
    "01. 持有 plain 时 plain.acquire(blocking=False)": "__未填写__",
    "02. 持有 reentrant 时再次 acquire 的结果": "__未填写__",
    "03. 退出后能否再次取得 plain": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from threading import Lock, RLock

plain = Lock()
reentrant = RLock()
with plain:
    plain_again = plain.acquire(blocking=False)
with reentrant:
    reentrant_again = reentrant.acquire(blocking=False)
    if reentrant_again:
        reentrant.release()
after = plain.acquire(blocking=False)
if after:
    plain.release()
actual_05 = (plain_again, reentrant_again, after)
print(*actual_05, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_05 = {
    "01. 持有 plain 时 plain.acquire(blocking=False)": "__未填写__",
    "02. 持有 reentrant 时再次 acquire 的结果": "__未填写__",
    "03. 退出后能否再次取得 plain": "__未填写__",
}

### 实验 5 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 5)

### 实验 5 原因解释

可重入锁解决哪种嵌套调用？为什么它仍可能参与两把锁之间的死锁？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 6：事件通知与协作停止

ready 只用于确认工作线程已进入，stop 控制继续。分别记录停止前和线程结束后的状态，不用固定休眠建立先后关系。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_06 = {
    "01. stop.set 前 stop.is_set()": "__未填写__",
    "02. join 后结果队列中的记录": "__未填写__",
    "03. join 后 thread.is_alive()": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from queue import Queue
from threading import Event, Thread

ready, stop = Event(), Event()
results = Queue()
def work():
    ready.set()
    if not stop.wait(timeout=5):
        raise TimeoutError("没有收到停止请求")
    results.put("cleaned")

thread = Thread(target=work)
thread.start()
try:
    if not ready.wait(timeout=5):
        raise TimeoutError("线程没有就绪")
    before = stop.is_set()
finally:
    stop.set()
    thread.join()
actual_06 = (before, results.get_nowait(), thread.is_alive())
print(*actual_06, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_06 = {
    "01. stop.set 前 stop.is_set()": "__未填写__",
    "02. join 后结果队列中的记录": "__未填写__",
    "03. join 后 thread.is_alive()": "__未填写__",
}

### 实验 6 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 6)

### 实验 6 原因解释

请求停止与已经停止有何区别？如果工作线程堵在永不返回的同步调用中，set 能强制让它退出吗？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 7：信号量名额与重复释放

逐次观察非阻塞取得与释放；只有成功取得过的名额才应释放。记录最后一次调用的实际情况。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_07 = {
    "01. 前三次 acquire 的结果": "__未填写__",
    "02. 第三次 release 的记录": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from threading import BoundedSemaphore

slots = BoundedSemaphore(2)
attempts = [slots.acquire(blocking=False) for _ in range(3)]
slots.release()
slots.release()
try:
    slots.release()
    release_record = "returned"
except Exception as error:
    release_record = type(error).__name__
actual_07 = (attempts, release_record)
print(*actual_07, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_07 = {
    "01. 前三次 acquire 的结果": "__未填写__",
    "02. 第三次 release 的记录": "__未填写__",
}

### 实验 7 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 7)

### 实验 7 原因解释

名额限制与每秒请求速率有何区别？BoundedSemaphore 相对普通 Semaphore 多检查什么？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 8：条件变量与共享缓冲区

生产者在同一把条件锁下修改 buffer 并通知，消费者使用 wait_for 检查条件。生产者先运行或后运行都不应改变数据约定。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_08 = {
    "01. 消费者放入 results 的结果": "__未填写__",
    "02. join 后的 buffer": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from queue import Queue
from threading import Condition, Thread

condition = Condition()
buffer = []
results = Queue()
def consume():
    with condition:
        if not condition.wait_for(lambda: bool(buffer), timeout=5):
            raise TimeoutError("缓冲区没有数据")
        results.put(buffer.pop())

thread = Thread(target=consume)
thread.start()
with condition:
    buffer.append("report")
    condition.notify()
thread.join()
actual_08 = (results.get_nowait(), buffer.copy())
print(*actual_08, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_08 = {
    "01. 消费者放入 results 的结果": "__未填写__",
    "02. join 后的 buffer": "__未填写__",
}

### 实验 8 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 8)

### 实验 8 原因解释

wait_for 等待时是否一直持锁？为什么被通知后仍需重新检查 buffer，而不能直接假设有数据？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 9：有界队列与完成通知

本题在单线程中观察队列接口；生产消费协调在讲义和异步实验中继续练习。逐项配对 get 与 task_done，最后再调用 join。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_09 = {
    "01. 队列满时 put_nowait 的记录": "__未填写__",
    "02. 取出的 first 和 second": "__未填写__",
    "03. 全部 task_done 后 join() 的返回结果": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from queue import Queue

jobs = Queue(maxsize=2)
jobs.put_nowait(10)
jobs.put_nowait(20)
try:
    jobs.put_nowait(30)
    record = "returned"
except Exception as error:
    record = type(error).__name__
first = jobs.get_nowait()
jobs.task_done()
second = jobs.get_nowait()
jobs.task_done()
returned = jobs.join()
actual_09 = (record, [first, second], returned)
print(*actual_09, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_09 = {
    "01. 队列满时 put_nowait 的记录": "__未填写__",
    "02. 取出的 first 和 second": "__未填写__",
    "03. 全部 task_done 后 join() 的返回结果": "__未填写__",
}

### 实验 9 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 9)

### 实验 9 原因解释

队列为空、任务计数结束、业务全部成功分别意味着什么？为什么 join 不是收集返回值？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 10：Future 的成功与异常

线程池保存工作函数的结果与异常。分别观察 result 与 done，不把结束状态直接解释为成功。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_10 = {
    "01. good.result()": "__未填写__",
    "02. bad.result() 的记录": "__未填写__",
    "03. bad.done()": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def convert(text):
    return int(text)

with ThreadPoolExecutor(max_workers=2) as pool:
    good = pool.submit(convert, "42")
    bad = pool.submit(convert, "invalid")
    value = good.result()
    try:
        bad.result()
        record = "returned"
    except Exception as error:
        record = type(error).__name__
    finished = bad.done()
actual_10 = (value, record, finished)
print(*actual_10, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_10 = {
    "01. good.result()": "__未填写__",
    "02. bad.result() 的记录": "__未填写__",
    "03. bad.done()": "__未填写__",
}

### 实验 10 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 10)

### 实验 10 原因解释

普通 Thread.join 与 Future.result 如何区别对待返回值和异常？不消费 Future 的异常会产生什么问题？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 11：完成顺序与输入位置

slow 必须等待 release；主线程收到第一个完成项后才释放它。用事件关系推理，不比较机器耗时。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_11 = {
    "01. as_completed 消费到的值顺序": "__未填写__",
    "02. 按输入 Future 位置再次读取的值": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Event

release = Event()
def slow():
    if not release.wait(timeout=5):
        raise TimeoutError("没有释放 slow")
    return "slow"
def fast():
    return "fast"

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(slow), pool.submit(fast)]
    completed = as_completed(futures)
    try:
        received = [next(completed).result()]
    finally:
        release.set()
    received.append(next(completed).result())
    input_order = [future.result() for future in futures]
actual_11 = (received, input_order)
print(*actual_11, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_11 = {
    "01. as_completed 消费到的值顺序": "__未填写__",
    "02. 按输入 Future 位置再次读取的值": "__未填写__",
}

### 实验 11 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 11)

### 实验 11 原因解释

结果关联为什么应使用编号或位置？如果两项都已结束，再创建 as_completed，是否还能要求同样的精确顺序？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 12：排队工作的取消与运行中工作的取消

线程池只有一个工作线程，running 已进入等待后才提交 queued。先尝试取消，最后释放运行中工作并退出线程池。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_12 = {
    "01. running.cancel()": "__未填写__",
    "02. queued.cancel()": "__未填写__",
    "03. queued.result() 的记录": "__未填写__",
    "04. running 最终的返回结果": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from threading import Event

started, release = Event(), Event()
def work():
    started.set()
    if not release.wait(timeout=5):
        raise TimeoutError("工作没有被释放")
    return "finished"

with ThreadPoolExecutor(max_workers=1) as pool:
    running = pool.submit(work)
    try:
        if not started.wait(timeout=5):
            raise TimeoutError("工作未启动")
        queued = pool.submit(lambda: "queued")
        cancel_running = running.cancel()
        cancel_queued = queued.cancel()
        try:
            queued.result()
            record = "returned"
        except Exception as error:
            record = type(error).__name__
    finally:
        release.set()
    value = running.result()
actual_12 = (cancel_running, cancel_queued, record, value)
print(*actual_12, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_12 = {
    "01. running.cancel()": "__未填写__",
    "02. queued.cancel()": "__未填写__",
    "03. queued.result() 的记录": "__未填写__",
    "04. running 最终的返回结果": "__未填写__",
}

### 实验 12 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 12)

### 实验 12 原因解释

Future 的取消能否强制停止已运行的线程？若改为 result(timeout=...)，等待超时与底层工作状态有何关系？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 13：spawn 子进程中的普通列表

阅读 source 中的完整独立脚本，再填写预测。run_script 会把源码写入临时 .py，使用当前解释器运行，读取 JSON 输出并清理临时目录；Notebook 不直接承担 spawn 的主模块。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_13 = {
    "01. 父进程的 original": "__未填写__",
    "02. 父进程从管道接收的内容": "__未填写__",
    "03. 子进程退出码": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from chapter10_checks import run_script

source = r"""
import json
import multiprocessing as mp

def work(items, sender):
    try:
        items.append("child")
        sender.send(items)
    finally:
        sender.close()

if __name__ == "__main__":
    context = mp.get_context("spawn")
    receiver, sender = context.Pipe(duplex=False)
    original = ["parent"]
    process = context.Process(target=work, args=(original, sender))
    process.start()
    sender.close()
    try:
        if not receiver.poll(10):
            raise TimeoutError("子进程没有返回")
        received = receiver.recv()
    finally:
        receiver.close()
        process.join(timeout=10)
        if process.is_alive():
            process.terminate()
            process.join()
        code = process.exitcode
        process.close()
    print(json.dumps([original, received, code]))
"""
actual_13 = tuple(run_script(source))
print(*actual_13, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_13 = {
    "01. 父进程的 original": "__未填写__",
    "02. 父进程从管道接收的内容": "__未填写__",
    "03. 子进程退出码": "__未填写__",
}

### 实验 13 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 13)

### 实验 13 原因解释

为什么工作入口要在可导入的文件顶层，启动动作却要放在主入口保护下？跨进程传输后的列表还共享普通内存吗？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 14：进程池的结果关联

本题同样运行临时完整脚本，明确使用 spawn。跟踪 map 的输入顺序；数据很小，仅用于行为观察，不用于宣称加速。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_14 = {
    "01. 进程池 map 按位置收集的 values": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
from chapter10_checks import run_script

source = r"""
import json
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

def square(number):
    return number * number

if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=2, mp_context=mp.get_context("spawn")) as pool:
        values = list(pool.map(square, [3, 1, 2]))
    print(json.dumps(values))
"""
actual_14 = (run_script(source),)
print(*actual_14, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_14 = {
    "01. 进程池 map 按位置收集的 values": "__未填写__",
}

### 实验 14 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 14)

### 实验 14 原因解释

为什么不把 Notebook 的局部 lambda 直接提交给标准进程池？这项微小计算为何不能证明多进程更快？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 15：协程对象的创建与执行

先调用协程函数保存 pending，再等待它。预测各阶段事件和结果；本单元格使用 Jupyter 的顶层 await。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_15 = {
    "01. await 前 before": "__未填写__",
    "02. await 后 events": "__未填写__",
    "03. 协程完成的 value": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
async def experiment_15():
    events = []
    async def work():
        events.append("body")
        return 12
    pending = work()
    before = events.copy()
    value = await pending
    return before, events, value

actual_15 = await experiment_15()
print(*actual_15, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_15 = {
    "01. await 前 before": "__未填写__",
    "02. await 后 events": "__未填写__",
    "03. 协程完成的 value": "__未填写__",
}

### 实验 15 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 15)

### 实验 15 原因解释

协程函数、协程对象和 Task 哪个在何时出现？为什么 Notebook 不在这个位置调用 asyncio.run？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 16：顺序 await 与独立任务

前半段依次等待两个调用；后半段建立两个任务，各任务记录开始后在同一事件处会合。只判断明确的顺序约束。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_16 = {
    "01. 顺序调用的 serial_events": "__未填写__",
    "02. 并发任务最早结束前的开始记录数量": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
import asyncio

async def experiment_16():
    serial_events = []
    async def serial_work(label):
        serial_events.append(label + " start")
        await asyncio.sleep(0)
        serial_events.append(label + " end")
    await serial_work("A")
    await serial_work("B")

    starts = []
    counts_at_end = []
    gate = asyncio.Event()
    async def concurrent_work(label):
        starts.append(label)
        if len(starts) == 2:
            gate.set()
        await gate.wait()
        counts_at_end.append(len(starts))
    async with asyncio.TaskGroup() as group:
        group.create_task(concurrent_work("A"))
        group.create_task(concurrent_work("B"))
    return serial_events, counts_at_end[0]

actual_16 = await experiment_16()
print(*actual_16, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_16 = {
    "01. 顺序调用的 serial_events": "__未填写__",
    "02. 并发任务最早结束前的开始记录数量": "__未填写__",
}

### 实验 16 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 16)

### 实验 16 原因解释

依次写两次 await 为什么没有使第二次调用提前开始？事件会合建立了什么可判断约束？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 17：无需暂停的 await

instant 没有任何等待点。只有在 await 返回后，主协程才设置 gate；另一个任务必须等 gate。预测事件顺序，不依赖任务工厂的启动时机。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_17 = {
    "01. 最终 events": "__未填写__",
    "02. instant 的返回结果": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
import asyncio

async def experiment_17():
    events = []
    gate = asyncio.Event()
    async def other():
        await gate.wait()
        events.append("other")
    async def instant():
        events.append("instant")
        return 7
    task = asyncio.create_task(other())
    events.append("before")
    value = await instant()
    events.append("after")
    gate.set()
    await task
    return events, value

actual_17 = await experiment_17()
print(*actual_17, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_17 = {
    "01. 最终 events": "__未填写__",
    "02. instant 的返回结果": "__未填写__",
}

### 实验 17 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 17)

### 实验 17 原因解释

这个例子中 instant 为什么能立即完成？await 的语法存在与实际交出执行机会为什么不能直接等同？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 18：gather 的位置顺序

slow 等待 fast 写入完成记录后才继续。分别记录真实结束顺序与 gather 返回位置，不把两者混为一谈。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_18 = {
    "01. completion": "__未填写__",
    "02. gather 返回的 values": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
import asyncio

async def experiment_18():
    gate = asyncio.Event()
    completion = []
    async def slow():
        await gate.wait()
        completion.append("slow")
        return "slow"
    async def fast():
        completion.append("fast")
        gate.set()
        return "fast"
    values = await asyncio.gather(slow(), fast())
    return completion, values

actual_18 = await experiment_18()
print(*actual_18, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_18 = {
    "01. completion": "__未填写__",
    "02. gather 返回的 values": "__未填写__",
}

### 实验 18 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 18)

### 实验 18 原因解释

为什么可以先结束后排在第二个结果位置？当用例编号与返回顺序不一致时应怎样关联？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 19：gather 失败后的其他成员

一个成员先确认 sibling 已运行再失败。捕获后检查 sibling 的状态，最后显式释放并等待它，避免本实验遗留后台任务。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_19 = {
    "01. gather 抛出的异常记录": "__未填写__",
    "02. 捕获后 sibling.done()": "__未填写__",
    "03. 最后 sibling 的返回值": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
import asyncio

async def experiment_19():
    started, release = asyncio.Event(), asyncio.Event()
    async def work():
        started.set()
        await release.wait()
        return "finished"
    async def fail():
        await started.wait()
        raise ValueError("failed")
    sibling = asyncio.create_task(work())
    try:
        await asyncio.gather(sibling, fail())
        record = "returned"
    except Exception as error:
        record = type(error).__name__
    finally:
        finished_before_release = sibling.done()
        release.set()
        value = await sibling
    return record, finished_before_release, value

actual_19 = await experiment_19()
print(*actual_19, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_19 = {
    "01. gather 抛出的异常记录": "__未填写__",
    "02. 捕获后 sibling.done()": "__未填写__",
    "03. 最后 sibling 的返回值": "__未填写__",
}

### 实验 19 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 19)

### 实验 19 原因解释

区分 gather 自身的失败规则与脚本退出时事件循环清理剩余任务的规则。为什么最后还要等待 sibling？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 20：TaskGroup 的失败与成员清理

先确认 waiting 已开始等待，再让 fail 抛出普通异常。记录异常组的成员类型以及离开任务组后的清理状态。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_20 = {
    "01. 捕获的成员类型名称": "__未填写__",
    "02. 任务组退出后的 cleaned": "__未填写__",
    "03. waiting_task.cancelled()": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
import asyncio

async def experiment_20():
    started = asyncio.Event()
    cleaned = []
    async def waiting():
        try:
            started.set()
            await asyncio.Event().wait()
        finally:
            cleaned.append("released")
    async def fail():
        await started.wait()
        raise ValueError("bad case")
    names = []
    try:
        async with asyncio.TaskGroup() as group:
            waiting_task = group.create_task(waiting())
            group.create_task(fail())
    except* ValueError as errors:
        names = sorted(type(error).__name__ for error in errors.exceptions)
    return names, cleaned, waiting_task.cancelled()

actual_20 = await experiment_20()
print(*actual_20, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_20 = {
    "01. 捕获的成员类型名称": "__未填写__",
    "02. 任务组退出后的 cleaned": "__未填写__",
    "03. waiting_task.cancelled()": "__未填写__",
}

### 实验 20 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 20)

### 实验 20 原因解释

为什么 TaskGroup 退出后可以检查清理记录？这是否也撤销了成员此前已完成的外部写入？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 21：协程竞态与 asyncio.Lock

两项无锁更新先读取旧值，再在事件处会合；另一组更新在异步锁里读取并写回。按明确的等待位置推导状态。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_21 = {
    "01. 无锁更新后的 count": "__未填写__",
    "02. 加锁更新后的 count": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
import asyncio

async def experiment_21():
    state = {"count": 0}
    arrived = []
    gate = asyncio.Event()
    async def unsafe():
        old = state["count"]
        arrived.append(True)
        if len(arrived) == 2:
            gate.set()
        await gate.wait()
        state["count"] = old + 1
    await asyncio.gather(unsafe(), unsafe())
    unsafe_value = state["count"]
    state["count"] = 0
    lock = asyncio.Lock()
    async def safe():
        async with lock:
            old = state["count"]
            await asyncio.sleep(0)
            state["count"] = old + 1
    await asyncio.gather(safe(), safe())
    return unsafe_value, state["count"]

actual_21 = await experiment_21()
print(*actual_21, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_21 = {
    "01. 无锁更新后的 count": "__未填写__",
    "02. 加锁更新后的 count": "__未填写__",
}

### 实验 21 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 21)

### 实验 21 原因解释

只有一个循环线程为什么仍可能丢失更新？如果把 asyncio.Lock 换成 threading.Lock，并在持锁时 await，会有什么问题？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 22：异步队列、结束标记与结果汇总

建立固定数量消费者，处理三项数据。结果按排序后比较，不要求消费者完成顺序固定。记录每项确认完成后的状态。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_22 = {
    "01. 排序后的 results": "__未填写__",
    "02. 任务组退出后的 jobs.empty()": "__未填写__",
    "03. 退出的消费者数量": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
import asyncio

async def experiment_22():
    jobs = asyncio.Queue(maxsize=2)
    results, stopped = [], []
    stop = object()
    async def consume():
        while True:
            item = await jobs.get()
            try:
                if item is stop:
                    stopped.append(True)
                    return
                await asyncio.sleep(0)
                results.append(item * 10)
            finally:
                jobs.task_done()
    async with asyncio.TaskGroup() as group:
        for _ in range(2):
            group.create_task(consume())
        for value in [1, 2, 3]:
            await jobs.put(value)
        for _ in range(2):
            await jobs.put(stop)
        await jobs.join()
    return sorted(results), jobs.empty(), len(stopped)

actual_22 = await experiment_22()
print(*actual_22, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_22 = {
    "01. 排序后的 results": "__未填写__",
    "02. 任务组退出后的 jobs.empty()": "__未填写__",
    "03. 退出的消费者数量": "__未填写__",
}

### 实验 22 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 22)

### 实验 22 原因解释

每个消费者为什么都要有结束机会？maxsize 限制排队项，消费者数量限制执行项，两者各自解决什么问题？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 23：异步超时与清理完成

工作先用 entered 确认已启动，然后等待不会被设置的事件。wait_for 超时后，观察工作状态与 finally 的记录，不测量精确时长。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_23 = {
    "01. wait_for 的异常记录": "__未填写__",
    "02. 捕获后 task.cancelled()": "__未填写__",
    "03. 捕获后的 cleaned": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
import asyncio

async def experiment_23():
    entered = asyncio.Event()
    cleaned = []
    async def work():
        try:
            entered.set()
            await asyncio.Event().wait()
        finally:
            cleaned.append("released")
    task = asyncio.create_task(work())
    await entered.wait()
    try:
        await asyncio.wait_for(task, timeout=0.01)
        record = "returned"
    except Exception as error:
        record = type(error).__name__
    return record, task.cancelled(), cleaned

actual_23 = await experiment_23()
print(*actual_23, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_23 = {
    "01. wait_for 的异常记录": "__未填写__",
    "02. 捕获后 task.cancelled()": "__未填写__",
    "03. 捕获后的 cleaned": "__未填写__",
}

### 实验 23 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 23)

### 实验 23 原因解释

wait_for 与 Future.result(timeout) 对底层工作的处理为何不同？为什么异步超时不是严格的强制终止时刻？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 24：显式取消与取消异常的继承

确认任务开始后发出取消并等待。记录请求、最终状态和清理；最后检查 asyncio.CancelledError 的继承关系。

### 运行前预测

阅读下一段实验代码，但先填写预测，不运行实验。

In [ ]:
prediction_24 = {
    "01. task.cancel() 的结果": "__未填写__",
    "02. 等待结束后 task.cancelled()": "__未填写__",
    "03. cleaned": "__未填写__",
    "04. issubclass(asyncio.CancelledError, Exception)": "__未填写__",
}

### 运行并观察

运行当前实验，按字段记录实际结果。

In [ ]:
import asyncio

async def experiment_24():
    entered = asyncio.Event()
    cleaned = []
    async def work():
        try:
            entered.set()
            await asyncio.Event().wait()
        finally:
            cleaned.append("released")
    task = asyncio.create_task(work())
    await entered.wait()
    requested = task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        pass
    return requested, task.cancelled(), cleaned, issubclass(asyncio.CancelledError, Exception)

actual_24 = await experiment_24()
print(*actual_24, sep="\n")

### 实际观察记录

记录本题实际结果，保留之前的预测；在原因解释区写出差异。

In [ ]:
observation_24 = {
    "01. task.cancel() 的结果": "__未填写__",
    "02. 等待结束后 task.cancelled()": "__未填写__",
    "03. cleaned": "__未填写__",
    "04. issubclass(asyncio.CancelledError, Exception)": "__未填写__",
}

### 实验 24 即时检查

只检查当前题，不改变预测或观察记录。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 24)

### 实验 24 原因解释

为什么先 cancel 再 await？普通 except Exception 与裸 except 的区别，会怎样影响取消传播和重试？

请在本单元格写自己的解释，尤其说明参与者、共享对象、等待条件和清理责任。

## 实验 25：实现线程安全计数器

实现 `LockedCounter(start=0)`。每个实例独立保存整数状态，`add(amount=1)` 完成一次完整加法并返回该次更新后的值，`snapshot()` 返回当前值。所有操作必须适用于多个线程同时调用；输入保证是整数，可以为负数或零。不修改调用方对象，不使用全局共享计数。检查器会交错调用并核对最终约束与每次返回值。

### 实现前方案

先写输入、返回、共享边界、失败与取消规则，以及准备验证的正常和边界场景。

In [ ]:
class LockedCounter:
    def __init__(self, start=0):
        raise NotImplementedError("请初始化独立状态与同步工具")

    def add(self, amount=1):
        raise NotImplementedError("请实现 add")

    def snapshot(self):
        raise NotImplementedError("请实现 snapshot")

### 自己的运行观察

在这里添加小例子和运行记录；先验证约束，再做本题即时检查。保留原预测和错误分析，不只记是否通过。

### 实验 25 即时检查

先运行实现定义，再运行当前检查。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 25)

### 实验 25 原因解释

说明必须共同保护的状态与操作。一次加法正确与整项接口线程安全有什么区别？

## 实验 26：实现保持输入顺序的线程池映射

实现 `thread_map_ordered(operation, items, *, max_workers=4)`。`items` 是有限、可只遍历一次的输入；返回新列表，位置与输入对应，不自行修改输入。调用同步 `operation(item)`，确实使用并发线程，运行中的操作不超过上限；数量足够且上限大于一时能够重叠运行。空输入返回空列表。`max_workers` 必须为正整数（不接受布尔值），否则抛出 `ValueError`。工作抛出的普通异常应传播给调用方；返回或抛出前确认本函数创建的工作已结束。允许使用标准库线程池；不要求为无限输入流提供固定内存。

### 实现前方案

先写输入、返回、共享边界、失败与取消规则，以及准备验证的正常和边界场景。

In [ ]:
def thread_map_ordered(operation, items, *, max_workers=4):
    raise NotImplementedError("请实现 thread_map_ordered")

### 自己的运行观察

在这里添加小例子和运行记录；先验证约束，再做本题即时检查。保留原预测和错误分析，不只记是否通过。

### 实验 26 即时检查

先运行实现定义，再运行当前检查。

In [ ]:
from chapter10_checks import check_experiment

check_experiment(globals(), 26)

### 实验 26 原因解释

如何同时满足输入顺序、并发上限与异常传播？为什么 with 退出等待可能晚于最早的失败？

## 实验 27：实现限制任务数量的异步映射

实现 `async_map_limited(operation, items, *, limit=3)`，它是异步函数。`operation(item)` 返回可等待对象，`items` 是有限、可只遍历一次的输入。返回与输入位置对应的新列表，不自行修改输入；空输入返回空列表。`limit` 必须为正整数（不接受布尔值），否则抛出 `ValueError`。同时进行的操作最多为 limit，任务数量也应随 limit 而不是输入数量增长；可以保存全部输入与结果，不要求整体 O(limit) 内存。输入足够时必须能并发推进。普通工作失败应向外传播（允许 ExceptionGroup），取消并等待本函数创建的其他工作；调用方取消时也必须清理并传播 asyncio.CancelledError。返回或抛出后不能遗留自己的运行任务。允许使用固定消费者、队列和 TaskGroup。

### 实现前方案

先写输入、返回、共享边界、失败与取消规则，以及准备验证的正常和边界场景。

In [ ]:
async def async_map_limited(operation, items, *, limit=3):
    raise NotImplementedError("请实现 async_map_limited")

### 自己的运行观察

在这里添加小例子和运行记录；先验证约束，再做本题即时检查。保留原预测和错误分析，不只记是否通过。

### 实验 27 即时检查

先运行实现定义，再运行当前检查。

In [ ]:
from chapter10_checks import check_async_experiment

await check_async_experiment(globals(), 27)

### 实验 27 原因解释

只用信号量包住每个操作是否限制了已创建的 Task 数量？失败与外部取消分别怎样收尾？

## 实验 28：实现保留取消语义的异步重试

实现 `async_retry(operation, *, attempts=3, retry_for=(TimeoutError,))`。operation 是无实参的异步可调用对象。attempts 是包含首次调用在内的最多尝试次数，必须是正整数（不接受布尔值），否则抛出 ValueError。只对 retry_for 中的普通异常重试；retry_for 保证为 Exception 子类构成的元组。成功立即返回，次数耗尽重新抛出最后那一个异常对象，未指定的异常立即传播。取消必须立即向外传播，不能被当作一次可重试业务失败。不添加真实休眠；本题只练异常与次数，不实现网络重试策略。允许空 retry_for。

### 实现前方案

先写输入、返回、共享边界、失败与取消规则，以及准备验证的正常和边界场景。

In [ ]:
async def async_retry(operation, *, attempts=3, retry_for=(TimeoutError,)):
    raise NotImplementedError("请实现 async_retry")

### 自己的运行观察

在这里添加小例子和运行记录；先验证约束，再做本题即时检查。保留原预测和错误分析，不只记是否通过。

### 实验 28 即时检查

先运行实现定义，再运行当前检查。

In [ ]:
from chapter10_checks import check_async_experiment

await check_async_experiment(globals(), 28)

### 实验 28 原因解释

普通失败与取消为什么应区别处理？如果实际操作是创建订单，在增加重试前还要确认什么？

## 全部复查

28 道题各自完成即时检查后再运行；保留预测、实际结果、实现和个人笔记。最后闭卷解释一项竞态、一项取消和一项资源隔离决策。

In [ ]:
from chapter10_checks import run_checks

await run_checks(globals())